# SFT Fine-Tuning: Qwen3-4B for STEM Tutoring

This notebook fine-tunes **Qwen3-4B** using Supervised Fine-Tuning (SFT) with QLoRA
for multi-domain STEM tutoring in Socratic style.

**Training pipeline stage:** 1 of 4 (SFT -> SimPO -> GRPO -> STaR)

**Requirements:** Google Colab with A100 GPU (40GB+ VRAM recommended)

**Domains:** Mathematics, Physics, Chemistry, Biology, Computer Science

In [ ]:
# Install dependencies
!pip install -q unsloth trl peft transformers datasets
!pip install -q accelerate bitsandbytes sentencepiece protobuf

# Login to HuggingFace (needed for pushing adapter after training)
from huggingface_hub import login
login()

In [ ]:
# ============================================================
# Configuration
# ============================================================

# Model
MODEL = "unsloth/Qwen3-4B-Instruct"
MAX_SEQ = 1024

# LoRA hyperparameters
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# Quantization
LOAD_IN_4BIT = True
BNB_4BIT_QUANT_TYPE = "nf4"

# Training hyperparameters
EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LR_SCHEDULER = "cosine"
WARMUP_STEPS = 50
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

# Checkpointing
SAVE_STEPS = 500
LOGGING_STEPS = 10

# Paths
DATASET_PATH = "training/data/combined_stem_balanced.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/MITS/checkpoints/sft_qwen3_4b"
VALIDATION_SPLIT = 0.05

# Domains for balanced sampling
DOMAINS = ["math", "physics", "chemistry", "biology", "cs"]

# Difficulty mapping (Russian labels → curriculum categories)
EASY_DIFFICULTIES = {"школьный"}
MEDIUM_DIFFICULTIES = {"базовый университетский"}
HARD_DIFFICULTIES = {"продвинутый", "олимпиадный"}

print(f"Model: {MODEL}")
print(f"LoRA rank: {LORA_R}, alpha: {LORA_ALPHA}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Epochs: {EPOCHS}, LR: {LEARNING_RATE}")
print(f"Dataset: {DATASET_PATH}")

In [2]:
# ============================================================
# Mount Google Drive (optional) and load dataset from HuggingFace
# ============================================================
import json
import os
from pathlib import Path
from collections import Counter

# Try mounting Drive for checkpoint persistence; fall back to local
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    print("Google Drive mounted successfully")
except Exception as e:
    print(f"Drive mount failed ({e}), using local storage")
    OUTPUT_DIR = "/content/checkpoints/sft_qwen3_4b"

# Load from HuggingFace Hub
from datasets import load_dataset

print("Loading SFT dataset from Siesher/mits-stem-training-data...")
hf_ds = load_dataset("Siesher/mits-stem-training-data", "sft")

train_records = [dict(r) for r in hf_ds["train"]]
val_records = [dict(r) for r in hf_ds["test"]]

print(f"Train: {len(train_records)}, Validation: {len(val_records)}")

# Analyze domain distribution
domain_counts = Counter(r.get("domain", "unknown") for r in train_records)
for domain, count in sorted(domain_counts.items()):
    print(f"  {domain}: {count} examples")

# Analyze difficulty distribution
difficulty_counts = Counter(r.get("difficulty", "unknown") for r in train_records)
for diff, count in sorted(difficulty_counts.items()):
    print(f"  {diff}: {count} examples")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\nCheckpoints will be saved to: {OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully
Loading SFT dataset from Siesher/mits-stem-training-data...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Train: 38330, Validation: 2018
  biology: 5322 examples
  chemistry: 6809 examples
  cs: 7020 examples
  math: 9566 examples
  physics: 9613 examples
  базовый университетский: 11320 examples
  олимпиадный: 7651 examples
  продвинутый: 7763 examples
  школьный: 11596 examples

Checkpoints will be saved to: /content/drive/MyDrive/MITS/checkpoints/sft_qwen3_4b


In [ ]:
# ============================================================
# Evaluate BASE model (before any training)
# ============================================================
# This establishes the baseline for all pipeline comparisons.
# Results are saved to Drive for cross-stage graph building.

import sys
sys.path.insert(0, "/content/drive/MyDrive/MITS")

from training.scripts.evaluate_stage import (
    load_eval_dataset, evaluate_with_model, save_report,
    append_summary_csv, print_comparison,
)
from unsloth import FastLanguageModel
import torch

EVAL_PATH = "/content/drive/MyDrive/MITS/eval_benchmark.jsonl"
REPORT_DIR = "/content/drive/MyDrive/MITS/evaluation/reports"
SUMMARY_CSV = os.path.join(REPORT_DIR, "summary.csv")

# Load base model (no LoRA)
print("Loading base Qwen3-4B for baseline evaluation...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=None,
)

# Evaluate
eval_problems = load_eval_dataset(EVAL_PATH)
baseline_results = evaluate_with_model(base_model, base_tokenizer, eval_problems)

# Save
save_report(baseline_results, "base", "colab", MODEL, REPORT_DIR)
append_summary_csv(baseline_results, "base", SUMMARY_CSV)
print_comparison(baseline_results, "base")

# Free memory for training
del base_model, base_tokenizer
torch.cuda.empty_cache()
print("\nBaseline saved. Memory freed for training.")

In [3]:
# ============================================================
# Load model with 4-bit quantization and apply LoRA
# ============================================================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # auto-detect
)

print(f"Model loaded: {MODEL}")
print(f"Model dtype: {model.dtype}")
print(f"Tokenizer vocab size: {len(tokenizer)}")

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Model loaded: unsloth/Qwen3-4B
Model dtype: torch.bfloat16
Tokenizer vocab size: 151669


Unsloth 2026.1.4 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.



Trainable parameters: 66,060,288 / 2,574,646,784 (2.57%)


In [4]:
# ============================================================
# Domain-balanced sampler
# ============================================================
import torch
from torch.utils.data import Sampler
from collections import defaultdict
import math


class DomainBalancedSampler(Sampler):
    """Sampler that ensures each batch contains examples from all 5 STEM domains.

    For each batch, we pick ceil(batch_size / num_domains) examples from each domain
    in a round-robin fashion, ensuring balanced domain exposure during training.
    """

    def __init__(self, dataset_records, batch_size, domains=None, seed=42):
        self.batch_size = batch_size
        self.domains = domains or DOMAINS
        self.seed = seed

        # Group indices by domain
        self.domain_indices = defaultdict(list)
        for idx, record in enumerate(dataset_records):
            domain = record.get("domain", "unknown")
            if domain in self.domains:
                self.domain_indices[domain].append(idx)
            else:
                # Assign unknown domains to a random bucket
                self.domain_indices[self.domains[idx % len(self.domains)]].append(idx)

        self.num_samples = sum(len(v) for v in self.domain_indices.values())
        self.per_domain_per_batch = max(1, self.batch_size // len(self.domains))

        print(f"DomainBalancedSampler: {self.num_samples} samples across {len(self.domains)} domains")
        for d in self.domains:
            print(f"  {d}: {len(self.domain_indices.get(d, []))} samples")

    def __iter__(self):
        rng = torch.Generator()
        rng.manual_seed(self.seed)

        # Shuffle indices within each domain
        shuffled = {}
        for domain in self.domains:
            indices = self.domain_indices[domain].copy()
            perm = torch.randperm(len(indices), generator=rng).tolist()
            shuffled[domain] = [indices[i] for i in perm]

        # Round-robin across domains
        domain_pointers = {d: 0 for d in self.domains}
        all_indices = []

        total_batches = self.num_samples // self.batch_size
        for _ in range(total_batches):
            batch = []
            for domain in self.domains:
                for _ in range(self.per_domain_per_batch):
                    if domain_pointers[domain] >= len(shuffled[domain]):
                        domain_pointers[domain] = 0  # wrap around
                    batch.append(shuffled[domain][domain_pointers[domain]])
                    domain_pointers[domain] += 1
            # Trim to exact batch size and shuffle within the batch
            batch = batch[:self.batch_size]
            perm = torch.randperm(len(batch), generator=rng).tolist()
            all_indices.extend([batch[i] for i in perm])

        return iter(all_indices)

    def __len__(self):
        return (self.num_samples // self.batch_size) * self.batch_size

In [5]:
# ============================================================                                                                                                                                                                                                    
# Curriculum dataset builder (HuggingFace Dataset format)
# ============================================================
from datasets import Dataset as HFDataset

SYSTEM_PROMPT = "Ты — сократический репетитор по математике, физике, химии, информатике и биологии. Помогай студентам, задавая наводящие вопросы."

def format_record(record):
    """Convert a record to formatted chat text."""
    messages = record.get("messages", [])
    if not messages:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": record.get("instruction", record.get("input", ""))},
            {"role": "assistant", "content": record.get("output", record.get("response", ""))},
        ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def build_curriculum_dataset(records, epoch=1):
    """Build HF Dataset filtered by curriculum epoch."""
    easy_medium = [
        r for r in records
        if r.get("difficulty", "") in EASY_DIFFICULTIES | MEDIUM_DIFFICULTIES
    ]
    hard = [
        r for r in records
        if r.get("difficulty", "") in HARD_DIFFICULTIES
    ]

    # Fallback if no labels matched
    if not easy_medium and not hard:
        print("WARNING: No difficulty labels matched, using all records.")
        easy_medium = records

    if epoch == 1:
        active = easy_medium
    elif epoch == 2:
        active = records
    else:
        active = easy_medium + hard * 2

    texts = [format_record(r) for r in active]
    print(f"Epoch {epoch}: {len(texts)} examples")
    return HFDataset.from_dict({"text": texts})


# Build validation set (all difficulties)
val_dataset = build_curriculum_dataset(val_records, epoch=2)
print(f"Validation: {len(val_dataset)} examples")

Epoch 2: 2018 examples
Validation: 2018 examples


In [6]:
from trl import SFTTrainer, SFTConfig
from datasets import Dataset as HFDataset
import torch

os.makedirs(OUTPUT_DIR, exist_ok=True)

DIFFICULTY_ORDER = {
    "школьный": 0,
    "базовый университетский": 1,
    "продвинутый": 2,
    "олимпиадный": 3,
}

sorted_records = sorted(
    train_records,
    key=lambda r: DIFFICULTY_ORDER.get(r.get("difficulty", ""), 1)
)
texts = [format_record(r) for r in sorted_records]
train_ds = HFDataset.from_dict({"text": texts})
print(f"Train: {len(train_ds)} examples, sorted by difficulty")

# >>> FIX: prevent gradient offloading <<<
if hasattr(model, 'hf_device_map'):
    model.hf_device_map = {'': 0}

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=10,
    save_steps=500,
    save_total_limit=3,
    bf16=True,
    optim="adamw_8bit",
    seed=42,
    report_to="none",
    max_seq_length=1024,
    dataset_text_field="text",
    packing=True,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_dataset,
    args=sft_config,
)

result = trainer.train()
print(f"\nDone! Loss: {result.training_loss:.4f}")

trainer.save_model(os.path.join(OUTPUT_DIR, "final"))

Train: 38330 examples, sorted by difficulty


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/38330 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=16):   0%|          | 0/38330 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/2018 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=16):   0%|          | 0/2018 [00:00<?, ? examples/s]

...

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 32,213 | Num Epochs = 3 | Total steps = 6,042
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.101000
20,0.809500
30,0.703400
40,0.621700
50,0.603600
60,0.576900


KeyboardInterrupt: 

In [ ]:
# ============================================================
# Evaluate on validation split
# ============================================================
from collections import defaultdict

print("Evaluating on held-out validation set...")

# Run evaluation
eval_results = trainer.evaluate()
print(f"\nOverall eval loss: {eval_results.get('eval_loss', 'N/A'):.4f}")
print(f"Eval perplexity: {math.exp(eval_results.get('eval_loss', 0)):.2f}")

# Per-domain evaluation
FastLanguageModel.for_inference(model)

domain_metrics = defaultdict(lambda: {"correct": 0, "total": 0, "total_loss": 0.0})

for record in val_records[:200]:  # sample for speed
    domain = record.get("domain", "unknown")
    messages = record.get("messages", [])
    if not messages:
        continue

    # Use all messages except the last assistant turn as input
    input_messages = [m for m in messages if m["role"] != "assistant"]
    prompt = tokenizer.apply_chat_template(
        input_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
        )

    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    domain_metrics[domain]["total"] += 1

    # Check if response follows Socratic style (contains a question)
    if "?" in generated:
        domain_metrics[domain]["correct"] += 1

print("\nPer-domain Socratic style adherence:")
for domain in DOMAINS:
    m = domain_metrics[domain]
    if m["total"] > 0:
        pct = 100 * m["correct"] / m["total"]
        print(f"  {domain}: {pct:.1f}% ({m['correct']}/{m['total']})")

# Log all metrics
metrics_path = os.path.join(OUTPUT_DIR, "eval_metrics.json")
with open(metrics_path, "w") as f:
    json.dump({
        "eval_results": eval_results,
        "domain_metrics": dict(domain_metrics),
        "curriculum_logs": all_logs,
    }, f, indent=2, default=str)
print(f"\nMetrics saved to {metrics_path}")

In [ ]:
# ============================================================
# Save LoRA adapter to Drive + push to HuggingFace
# ============================================================

# Save final adapter
final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"Final LoRA adapter saved to {final_adapter_path}")

# Save training config for reproducibility
config_to_save = {
    "stage": "sft",
    "pipeline": "SFT -> GSPO (curriculum) -> RAFT++ -> AdaSTaR -> DPO",
    "pipeline_position": "1 of 5",
    "model": MODEL,
    "max_seq_length": MAX_SEQ,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "target_modules": TARGET_MODULES,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "scheduler": LR_SCHEDULER,
    "warmup_steps": WARMUP_STEPS,
    "quantization": "4bit_nf4",
    "train_samples": len(train_records),
    "val_samples": len(val_records),
}
config_path = os.path.join(OUTPUT_DIR, "training_config.json")
with open(config_path, "w") as f:
    json.dump(config_to_save, f, indent=2)
print(f"Config saved to {config_path}")

# Push to HuggingFace Hub
PUSH_TO_HUB = True
HF_REPO_ID = "Siesher/mits-qwen3-4b-sft"

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, private=True)
    tokenizer.push_to_hub(HF_REPO_ID, private=True)
    print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

print("\nDone! SFT adapter ready for GSPO (next stage).")

In [ ]:
# ============================================================
# Post-Training Evaluation on MITS Benchmark
# ============================================================
import sys
sys.path.insert(0, "/content/drive/MyDrive/MITS")

from training.scripts.evaluate_stage import (
    load_eval_dataset, evaluate_with_model, save_report,
    append_summary_csv, print_comparison,
)

STAGE = "sft"
EVAL_PATH = "/content/drive/MyDrive/MITS/eval_benchmark.jsonl"
REPORT_DIR = "/content/drive/MyDrive/MITS/evaluation/reports"
SUMMARY_CSV = os.path.join(REPORT_DIR, "summary.csv")

# Load baseline for comparison
import json, glob
baseline = None
base_reports = sorted(glob.glob(os.path.join(REPORT_DIR, "stage_base_*.json")))
if base_reports:
    with open(base_reports[-1], encoding="utf-8") as f:
        baseline = json.load(f).get("results")

# Evaluate
eval_problems = load_eval_dataset(EVAL_PATH)
results = evaluate_with_model(model, tokenizer, eval_problems)

# Save & compare
save_report(results, STAGE, "colab", HF_REPO_ID, REPORT_DIR)
append_summary_csv(results, STAGE, SUMMARY_CSV)
print_comparison(results, STAGE, baseline)